# SMS Spam Filter: walkthrough

This notebook walks through how the spam filter works. The reusable code lives in
[`spam_filter.py`](spam_filter.py), and the clickable demo is [`app.py`](app.py).

**Steps**
1. Load the data
2. Train the model
3. Measure accuracy on messages it has never seen
4. Look at what it gets wrong
5. Try it on new messages
6. Launch the Gradio demo

In [1]:
import pandas as pd

import spam_filter

pd.set_option("max_colwidth", 200)

## 1. Load the data

The [UCI SMS Spam Collection](https://archive.ics.uci.edu/dataset/228/sms+spam+collection) has 5,572
English text messages, each labelled `ham` (a real message) or `spam`. About 13% are spam.

In [2]:
sms_text_df = spam_filter.load_data()
sms_text_df.head()

,label,text_message
0,ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives around here though"


In [3]:
sms_text_df["label"].value_counts()

label
ham     4825
spam     747
Name: count, dtype: int64

## 2. Train the model

`sms_classification` splits the data (67% for training, 33% held back for testing) and fits a
pipeline with two steps:

- **TF-IDF vectorizer**: turns each message into numbers based on which words it uses and how
  distinctive those words are.
- **Linear Support Vector Classifier**: learns a boundary between the word patterns of spam and real messages.

In [4]:
text_clf = spam_filter.sms_classification(sms_text_df)
text_clf

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidf', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](2,)","['ham','spam']"
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True


## 3. Measure accuracy

The model is scored on the 1,839 test messages it never saw during training.
In the confusion matrix, rows are the true label and columns are the model's prediction (`ham`, then `spam`).

In [5]:
results = spam_filter.evaluate(text_clf, sms_text_df)
print(f"Test accuracy: {results['accuracy']:.4f}")
print(results["confusion_matrix"])
print(results["report"])

Test accuracy: 0.9897
[[1586    7]
 [  12  234]]
              precision    recall  f1-score   support

         ham      0.992     0.996     0.994      1593
        spam      0.971     0.951     0.961       246

    accuracy                          0.990      1839
   macro avg      0.982     0.973     0.978      1839
weighted avg      0.990     0.990     0.990      1839



For comparison, the original bootcamp solution removed common English "stop words" (such as *the*, *and*, *you*).
Keeping them scores slightly higher, so the app keeps them.

In [6]:
with_stop_words_removed = spam_filter.sms_classification(sms_text_df, stop_words="english")
print(f"Test accuracy with stop words removed: {spam_filter.evaluate(with_stop_words_removed, sms_text_df)['accuracy']:.4f}")

Test accuracy with stop words removed: 0.9886


## 4. What it gets wrong

- **Missed spam (`spam -> ham`)**: chatty or flirty spam that sounds like a friend, jokes, and spam
  without obvious words like *free*, *win* or *claim*.
- **False alarms (`ham -> spam`)**: short, casual real messages, especially ones that mention
  *calls*, *texts*, *chat* or being *free*.

In [7]:
results["errors"]

,actual,predicted,text_message
439,ham,spam,"Hi hope u get this txt~journey hasnt been gd,now about 50 mins late I think."
3864,spam,ham,"Oh my god! I've found your number again! I'm so glad, text me back xafter this msgs cst std ntwk chg £1.50"
2402,spam,ham,Babe: U want me dont u baby! Im nasty and have a thing 4 filthyguys. Fancy a rude time with a sexy bitch. How about we go slo n hard! Txt XXX SLO(4msgs)
84,ham,spam,Yup next stop.
2663,spam,ham,"Hello darling how are you today? I would love to have a chat, why dont you tell me what you look like and what you are in to sexy?"
751,spam,ham,"Do you realize that in about 40 years, we'll have thousands of old ladies running around with tattoos?"
3360,spam,ham,Sorry I missed your call let's talk when you have the time. I'm on 07090201529
2770,spam,ham,Burger King - Wanna play footy at a top stadium? Get 2 Burger King before 1st Sept and go Large or Super with Coca-Cola and walk out a winner
1988,ham,spam,No calls..messages..missed calls
731,spam,ham,Email AlertFrom: Jeri StewartSize: 2KBSubject: Low-cost prescripiton drvgsTo listen to email call 123


## 5. Try it on new messages

In [8]:
new_messages = [
    "You are a lucky winner of $5000!",
    "You won 2 free tickets to the Super Bowl.",
    "You won 2 free tickets to the Super Bowl text us to claim your prize.",
    "Thanks for registering. Text 4343 to receive free updates on medicare.",
    "Can you send me the report before the 3pm meeting?",
]
for text in new_messages:
    print(spam_filter.sms_prediction(text, text_clf))

The text message: "You are a lucky winner of $5000!", is not spam.
The text message: "You won 2 free tickets to the Super Bowl.", is spam.
The text message: "You won 2 free tickets to the Super Bowl text us to claim your prize.", is spam.
The text message: "Thanks for registering. Text 4343 to receive free updates on medicare.", is spam.
The text message: "Can you send me the report before the 3pm meeting?", is not spam.


Note the first message: an obvious prize scam is labelled "not spam". Spam in this dataset usually
asks you to text or call a number, so a scam without one can slip through.

## 6. Save the model and launch the demo

`python spam_filter.py` retrains the model and saves it to `model/spam_filter.joblib`, which the app loads.
To open the demo, run `python app.py` from a terminal, or uncomment the last line below.

In [9]:
from app import demo

# demo.launch()